In [1]:
!python3 -m pip install transformers

  Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 11.3 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 8.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 10.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 10.1 MB/s  0:00:00 eta 0:00:01
Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl (447 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [transformers] [transformers]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


In [79]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()
text = "Rose loves sweet fresh fruits and  vegetables. She loves  to  eat"
inputs = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits



Loading weights: 100%|██████████████████████████████████████████████████████████████████████| 148/148 [00:00<00:00, 18574.42it/s]


In [80]:
logits.shape

torch.Size([1, 15, 50257])

In [81]:
logits

tensor([[[ -31.2852,  -30.6273,  -34.2245,  ...,  -39.2340,  -38.1298,
           -31.6271],
         [ -74.0125,  -75.1880,  -81.0360,  ...,  -81.6594,  -82.0551,
           -76.6323],
         [ -84.9573,  -86.9921,  -93.2764,  ...,  -91.8847,  -94.9563,
           -88.3962],
         ...,
         [-110.2911, -111.2673, -114.5106,  ..., -117.3102, -116.9686,
          -113.5080],
         [ -75.8431,  -77.4616,  -80.0088,  ...,  -84.4123,  -81.4616,
           -77.6193],
         [-106.9341, -112.4048, -116.9172,  ..., -120.0759, -119.6651,
          -111.8896]]])

In [82]:
next_token_logits = logits[:, -1, :]

In [83]:
next_token_logits.shape

torch.Size([1, 50257])

In [84]:
next_token_logits

tensor([[-106.9341, -112.4048, -116.9172,  ..., -120.0759, -119.6651,
         -111.8896]])

In [85]:
probs = torch.softmax(next_token_logits, dim=-1)


In [86]:
probs

tensor([[9.3909e-03, 3.9520e-05, 4.3362e-07,  ..., 1.8421e-08, 2.7778e-08,
         6.6154e-05]])

In [87]:
top_k = torch.topk(probs, k=5)

In [88]:
top_k

torch.return_types.topk(
values=tensor([[0.0924, 0.0593, 0.0541, 0.0513, 0.0388]]),
indices=tensor([[  13, 4713,  220,  290,  606]]))

In [89]:
for idx, score in zip(top_k.indices[0], top_k.values[0]):
    predicted_word = tokenizer.decode([idx])
    print(f"{predicted_word!r} : {float(score):.4f}")

'.' : 0.0924
' fresh' : 0.0593
' ' : 0.0541
' and' : 0.0513
' them' : 0.0388


In [92]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to("cuda" if torch.cuda.is_available() else "cpu")

prompt = "Can you write a short beautiful love story?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(**inputs, max_new_tokens=1)
print(tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights: 100%|██████████████████████████████████████████████████████████████████████| 282/282 [00:00<00:00, 20338.99it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


The
